# Bloque de codificación tipo Hamming

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
from time import perf_counter
#Prueba de que el algoritmo funciona
# Función para codificar un nibble (4 bits de datos)

def h74_encode_nibble(nib):
    """Codificador Hamming (7,4) — solo codifica (sin decodificar)."""
    b3 = (nib >> 3) & 1
    b2 = (nib >> 2) & 1
    b1 = (nib >> 1) & 1
    b0 = (nib >> 0) & 1

    c = np.zeros(7, dtype=np.uint8)
    # Datos
    c[2], c[4], c[5], c[6] = b3, b2, b1, b0
    # Paridades (posiciones 1,2,4)
    c[0] = c[2] ^ c[4] ^ c[6]  # p1 cubre 1,3,5,7
    c[1] = c[2] ^ c[5] ^ c[6]  # p2 cubre 2,3,6,7
    c[3] = c[4] ^ c[5] ^ c[6]  # p4 cubre 4,5,6,7
    return c

def bytes_to_h74_bitstream(data):
    """Convierte lista de enteros (0–255) a bitstream Hamming(7,4)."""
    out = []
    for byte in data:
        hi = (byte >> 4) & 0xF
        lo = byte & 0xF
        out.extend(h74_encode_nibble(hi))
        out.extend(h74_encode_nibble(lo))
    return np.array(out, dtype=np.uint8)


def medir_tiempo(data, repeticiones=200):
    """Mide tiempo promedio de codificación Hamming(7,4)."""
    t0 = perf_counter()
    for _ in range(repeticiones):
        _ = bytes_to_h74_bitstream(data)
    t1 = perf_counter()
    print(f"Tiempo promedio: {(t1 - t0)/repeticiones*1e3:.4f} ms por corrida")


bpm_filtrado = [72, 75, 80, 90, 95]  # datos de ejemplo
print("Datos de entrada:", bpm_filtrado)

bits_codificados = bytes_to_h74_bitstream(bpm_filtrado)
print("Bits codificados (primeros 28 bits):", bits_codificados[:28], "...")
print("Tamaño total del bitstream:", len(bits_codificados), "bits")

# Medición de tiempo
medir_tiempo(bpm_filtrado)

Datos de entrada: [72, 75, 80, 90, 95]
Bits codificados (primeros 28 bits): [1 0 0 1 1 0 0 1 1 1 0 0 0 0 1 0 0 1 1 0 0 0 1 1 0 0 1 1] ...
Tamaño total del bitstream: 70 bits
Tiempo promedio: 0.0276 ms por corrida


# 7.4

In [11]:
def h74_encode_nibble(nib:int):
    b3 = (nib >> 3) & 1; b2 = (nib >> 2) & 1; b1 = (nib >> 1) & 1; b0 = (nib >> 0) & 1 #separa los 4 bits del nibble
    c = [0]*7 #lista vacía donde se guardará la palabra hamming
    c[2], c[4], c[5], c[6] = b3, b2, b1, b0 # asignación de posiciones
    c[0] = c[2] ^ c[4] ^ c[6] #cálculo de los bits de paridad
    c[1] = c[2] ^ c[5] ^ c[6]
    c[3] = c[4] ^ c[5] ^ c[6]
    return c #devuelve los bits codificados

def bytes_to_bits_h74(u8_list):
    out=[]
    for b in u8_list:
        hi=(b>>4)&0xF; lo=b&0xF #divide cada byte en dos nibbles
        out += h74_encode_nibble(hi) #aplica el codificador hamming a cada nibble
        out += h74_encode_nibble(lo)
    return np.array(out, dtype=np.uint8) #concatena los resultados

def verificar_h74(bits):
    assert len(bits) % 7 == 0 #verifica que los bloques sean múltiplos de 7
    malos=0  #sumador de bloques malos
    for i in range(0,len(bits),7):
        c=bits[i:i+7] #recorre los bits de 7 en 7 recalcula paridades (síndrome)
        s1=c[0]^c[2]^c[4]^c[6]
        s2=c[1]^c[2]^c[5]^c[6]
        s4=c[3]^c[4]^c[5]^c[6]
        if (s1|s2|s4)!=0: malos+=1
    return (len(bits)//7, malos)

def tiempo_stats_h74(data_bytes, repeticiones=200):
    ts=[]
    for _ in range(repeticiones):
        t0=perf_counter(); _=bytes_to_bits_h74(data_bytes); t1=perf_counter()
        ts.append(t1-t0)
    ts=np.array(ts); return ts.mean()


In [12]:
#entrada de datos
archivo_excel = Path("Documentacion/Avance 1/datos_filtrados/filtro_mediana_w5.xlsx")
columna = "Filtrado_Mediana"

df = pd.read_excel(archivo_excel, engine="openpyxl") 
serie = df[columna] if isinstance(columna, str) else df.iloc[:, columna]
data_u8 = np.clip(np.nan_to_num(serie.to_numpy(), nan=0.0), 0, 255).astype(np.uint8) 

print("Bytes listos:", len(data_u8))

Bytes listos: 127


In [15]:
bits_cod = bytes_to_bits_h74(data_u8)  
bits_cod = np.asarray(bits_cod, dtype=np.uint8).reshape(-1)  # vector columna 0/1
esperados = 14 * len(data_u8)

assert bits_cod.size == esperados, (
    f"Largo inesperado: {bits_cod.size} vs {esperados} (14 bits por byte). "
)
print(f"- Bytes entrada         : {len(data_u8)}")
print(f"- Bits codificados (7,4): {bits_cod.size} (esperado = {esperados})")

- Bytes entrada         : 127
- Bits codificados (7,4): 1778 (esperado = 1778)


In [16]:
# Exportar los bits codificados para el bloque de Modulación en matlab

salida_dir = Path("Matlab")
salida_dir.mkdir(parents=True, exist_ok=True)

ruta_txt = salida_dir / "bits_codificados_h74.txt"
ruta_npy = salida_dir / "bits_codificados_h74.npy"

# Guardar como texto (una columna de 0/1)
np.savetxt(ruta_txt, bits_cod.astype(np.uint8), fmt="%d")
# Guardar también en binario numpy (opcional)
np.save(ruta_npy, bits_cod.astype(np.uint8))



In [17]:
mean_s = tiempo_stats_h74(data_u8, repeticiones=200)

print(f"Archivo: {archivo_excel.name}")
print(f"Bytes de entrada: {len(data_u8)} | Bits codificados: {len(bits_cod)} (esperado: {14*len(data_u8)})")
print(f"Tiempo promedio (200): {mean_s*1e3:.4f} ms")


Archivo: filtro_mediana_w5.xlsx
Bytes de entrada: 127 | Bits codificados: 1778 (esperado: 1778)
Tiempo promedio (200): 0.4074 ms


# 15.11
El código Hamming (15,11) trabaja con bloques de 11 bits, no con bytes (8 bits).

In [18]:
# Utilidades de bits
def bytes_to_bitstream_MSBfirst(u8: np.ndarray) -> np.ndarray: 
    u8 = np.asarray(u8, dtype=np.uint8) #arreglo NumPy de enteros de 8 bits
    out = np.unpackbits(u8, bitorder='big')
    return out.astype(np.uint8)

def chunk11_with_padding(bits: np.ndarray) -> np.ndarray: #Convertir toda la señal (lista de bytes) a una secuencia continua de bits (bitstream)
    """Parte el bitstream en bloques de 11 bits, con padding de ceros al final."""
    n = bits.size # Calcula cuántos bits hay en total
    r = n % 11 # Calcula el residuo al dividir entre 11, es decir, cuántos bits “sobran”
    if r != 0:
        pad = np.zeros(11 - r, dtype=np.uint8) # Crea un arreglo con los ceros necesarios para completar el último bloque.
        bits = np.concatenate([bits, pad])
    return bits.reshape(-1, 11)


In [19]:
# se calculan los bits de paridad y se forman los 15 bits codificados
# Paridades en 1,2,4,8; datos en 3,5,6,7,9–15

def h1511_encode_block(bits11):
    b10,b9,b8,b7,b6,b5,b4,b3,b2,b1,b0 = map(int, bits11) #asignar posiciones
    c = [0]*15 # lista de 15 ceros que contendrá los bits codificados
    # Datos
    c[2], c[4], c[5], c[6], c[8], c[9], c[10], c[11], c[12], c[13], c[14] = \
        b10,b9,b8,b7,b6,b5,b4,b3,b2,b1,b0 
    # cálculo de los bits de paridad
    c[0] = c[2]^c[4]^c[6]^c[8]^c[10]^c[12]^c[14]
    c[1] = c[2]^c[5]^c[6]^c[9]^c[10]^c[13]^c[14]
    c[3] = c[4]^c[5]^c[6]^c[11]^c[12]^c[13]^c[14]
    c[7] = c[8]^c[9]^c[10]^c[11]^c[12]^c[13]^c[14]
    return np.array(c, dtype=np.uint8)

def bitstream_to_h1511(bits: np.ndarray) -> np.ndarray:
    """Codifica bitstream arbitrario en palabras Hamming (15,11)."""
    bloques = chunk11_with_padding(bits)
    codificados = [h1511_encode_block(b) for b in bloques] # llama a la primer función para cada bloque
    return np.concatenate(codificados).astype(np.uint8)


In [20]:
def verificar_h1511(bitstream15):
    """Verifica que cada bloque tenga síndrome (0,0,0,0).""" #recorre todos los bloques de 15 bits y verifica las paridades
    assert bitstream15.size % 15 == 0 #revisa que el tamaño total del arreglo sea múltiplo de 15
    malos = 0 #contador para malos codificados
    for i in range(0, bitstream15.size, 15):
        c = bitstream15[i:i+15] #cálculo de los bits de síndrome
        s1 = c[0]^c[2]^c[4]^c[6]^c[8]^c[10]^c[12]^c[14]
        s2 = c[1]^c[2]^c[5]^c[6]^c[9]^c[10]^c[13]^c[14]
        s4 = c[3]^c[4]^c[5]^c[6]^c[11]^c[12]^c[13]^c[14]
        s8 = c[7]^c[8]^c[9]^c[10]^c[11]^c[12]^c[13]^c[14]
        if (s1|s2|s4|s8)!=0:
            malos += 1
    total = bitstream15.size // 15
    return total, malos


In [21]:
def tiempo_stats_h1511(data_bytes_u8, repeticiones=200):
    """Mide el tiempo promedio de codificación Hamming (15,11)."""
    bits = bytes_to_bitstream_MSBfirst(data_bytes_u8)
    ts=[] #lista vacía donde se guardan los tiempos de cada ejecución
    for _ in range(repeticiones): #proceso de medición repetida
        t0 = perf_counter()
        _ = bitstream_to_h1511(bits)
        t1 = perf_counter()
        ts.append(t1 - t0)
    ts = np.array(ts) 
    return ts.mean(), ts.std() #tiempo promedio y desv estándar


In [22]:
# lectura de los datos reales
INFILE = Path("Documentacion/Avance 1/datos_filtrados/filtro_mediana_w5.xlsx")
COL = "Filtrado_Mediana"

df = pd.read_excel(INFILE, engine="openpyxl")
serie = df[COL]
data_u8 = np.clip(np.nan_to_num(serie.to_numpy(), nan=0.0), 0, 255).astype(np.uint8)

#Codificación Hamming (15,11)
bits_in   = bytes_to_bitstream_MSBfirst(data_u8)
bits_1511 = bitstream_to_h1511(bits_in)

# verificación
total, con_error = verificar_h1511(bits_1511)


print(f"Archivo: {INFILE.name}")
print(f"Bytes de entrada: {len(data_u8)} | Bits codificados: {len(bits_1511)}")
print(f"Bloques válidos: {total-con_error}/{total}")



Archivo: filtro_mediana_w5.xlsx
Bytes de entrada: 127 | Bits codificados: 1395
Bloques válidos: 93/93


In [23]:
mean_s, std_s = tiempo_stats_h1511(data_u8, repeticiones=200)
print(f"Tiempo promedio (200 corridas): {mean_s*1e3:.4f} ms")


Tiempo promedio (200 corridas): 0.3724 ms


In [24]:
try:
    bits_1511
except NameError:
    bits_in = bytes_to_bitstream_MSBfirst(data_u8)
    bits_1511 = bitstream_to_h1511(bits_in)

Path("Matlab").mkdir(parents=True, exist_ok=True)
np.savetxt("Matlab/bits_codificados_h1511.txt",
           np.asarray(bits_1511, np.uint8).reshape(-1), fmt="%d")